In [1]:
import os
os.system("pip install ultralytics -q")
os.system("git clone https://github.com/abdur75648/End-To-End-Urdu-OCR-WebApp.git")


# Find correct path
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        if f == "model.py":
            print(os.path.join(root, f))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00


Cloning into 'End-To-End-Urdu-OCR-WebApp'...


/kaggle/working/End-To-End-Urdu-OCR-WebApp/model.py


In [2]:
!bash /kaggle/working/End-To-End-Urdu-OCR-WebApp/download_files.sh


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1364  100  1364    0     0   6825      0 --:--:-- --:--:-- --:--:--  6854
100 2541k  100 2541k    0     0  6075k      0 --:--:-- --:--:-- --:--:-- 6075k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1358  100  1358    0     0   7407      0 --:--:-- --:--:-- --:--:--  7420
100 2480k  100 2480k    0     0  6455k      0 --:--:-- --:--:-- --:--:-- 24.7M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1370  100  1370    0     0   8593      0 --:--:-- --:--:-- --:--:--  8616
100 2319k  100 2319k    0     0  6599k      0 --:--:-- --:--:-- --:--:-- 6599k
  % Total    % Received % Xferd  Average Speed   Tim

In [3]:
import os, sys, torch
from PIL import Image

sys.path.append("/kaggle/working/End-To-End-Urdu-OCR-WebApp")
from read import text_recognizer
from model import Model
from utils import CTCLabelConverter
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# Supported file types
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

#Model Paths
GLYPHS_PATH    = "/kaggle/working/End-To-End-Urdu-OCR-WebApp/UrduGlyphs.txt"
RECOGNIZER_PTH = "/kaggle/working/best_norm_ED.pth"
DETECTOR_PT    = "/kaggle/working/yolov8m_UrduDoc.pt"

In [5]:
def load_models(device):
    with open(GLYPHS_PATH, "r", encoding="utf-8") as f:
        content = "".join(line.strip("\n") for line in f.readlines()) + " "
    converter = CTCLabelConverter(content)
    recognition_model = Model(num_class=len(converter.character), device=device)
    recognition_model = recognition_model.to(device)
    recognition_model.load_state_dict(torch.load(RECOGNIZER_PTH, map_location=device))
    recognition_model.eval()
    detection_model = YOLO(DETECTOR_PT)
    return detection_model, recognition_model, converter

In [6]:
def ocr_image(image_path, detection_model, recognition_model, converter, device):
    input_image = Image.open(image_path).convert("RGB")
    results = detection_model.predict(
        source=input_image, conf=0.2, imgsz=1280,
        save=False, nms=True, device=device
    )
    bounding_boxes = results[0].boxes.xyxy.cpu().numpy().tolist()
    bounding_boxes.sort(key=lambda x: x[1])   # top-to-bottom reading order
    texts = []
    for box in bounding_boxes:
        cropped = input_image.crop(box)
        texts.append(text_recognizer(cropped, recognition_model, converter, device))
    return "\n".join(texts)

In [7]:
# Resume Functioanlity

def get_progress_file(output_txt):
    base, _ = os.path.splitext(output_txt)
    return base + ".progress"

def load_progress(progress_file):
    if not os.path.exists(progress_file):
        return set()
    with open(progress_file, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

def save_progress(progress_file, filename):
    with open(progress_file, "a", encoding="utf-8") as f:
        f.write(filename + "\n")

In [8]:
import re

def natural_sort_key(s):
    """Split string into list of text and number chunks for natural sorting."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

In [9]:
def process_directory(image_dir, output_dir="/kaggle/working"):
    image_dir     = os.path.abspath(image_dir)
    dir_name      = os.path.basename(image_dir.rstrip(os.sep))
    os.makedirs(output_dir, exist_ok=True)
    output_txt    = os.path.join(output_dir, dir_name + ".txt")
    progress_file = os.path.join(output_dir, dir_name + ".progress")

    all_images = sorted(
        (f for f in os.listdir(image_dir)
         if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS),
        key=natural_sort_key
    )
    if not all_images:
        print(f"No supported images found in: {image_dir}")
        return

    done      = load_progress(progress_file)
    remaining = [f for f in all_images if f not in done]

    print(f"Directory  : {image_dir}")
    print(f"Output file: {output_txt}")
    print(f"Total pages: {len(all_images)}  |  Done: {len(done)}  |  Remaining: {len(remaining)}")

    if not remaining:
        print("All pages already processed. Nothing to do.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print("Loading models ...")
    detection_model, recognition_model, converter = load_models(device)
    print("Models loaded.\n")

    with open(output_txt, "a", encoding="utf-8") as out_f:
        for idx, filename in enumerate(remaining, start=1):
            image_path = os.path.join(image_dir, filename)
            print(f"[{idx}/{len(remaining)}] Processing: {filename}")
            try:
                text = ocr_image(image_path, detection_model, recognition_model, converter, device)
                out_f.write(f"### {filename} ###\n")
                out_f.write(text + "\n\n")
                out_f.flush()
                save_progress(progress_file, filename)
                print(f"           Done  ({len(text.splitlines())} lines recognised)")
            except Exception as e:
                print(f"           ERROR: {e}  -- skipping this page")

    print(f"\nFinished! Text saved to: {output_txt}")

In [10]:
for i in range(1,29):
    process_directory(f"/kaggle/input/datasets/ishahzaibkhan/gugtugu-images/Guftagu_{i}")

Directory  : /kaggle/input/datasets/ishahzaibkhan/gugtugu-images/Guftagu_1
Output file: /kaggle/working/Guftagu_1.txt
Total pages: 297  |  Done: 0  |  Remaining: 297
Using device: cuda
Loading models ...
Models loaded.

[1/297] Processing: page_1.jpg

0: 1280x832 (no detections), 95.8ms
Speed: 28.9ms preprocess, 95.8ms inference, 5.0ms postprocess per image at shape (1, 3, 1280, 832)
           Done  (0 lines recognised)
[2/297] Processing: page_2.jpg

0: 1280x1280 4 texts, 106.0ms
Speed: 9.4ms preprocess, 106.0ms inference, 34.2ms postprocess per image at shape (1, 3, 1280, 1280)
           Done  (4 lines recognised)
[3/297] Processing: page_3.jpg

0: 1248x1280 14 texts, 64.9ms
Speed: 9.1ms preprocess, 64.9ms inference, 1.4ms postprocess per image at shape (1, 3, 1248, 1280)
           Done  (14 lines recognised)
[4/297] Processing: page_4.jpg

0: 1280x1280 3 texts, 64.1ms
Speed: 9.5ms preprocess, 64.1ms inference, 1.4ms postprocess per image at shape (1, 3, 1280, 1280)
           Don

In [11]:
import shutil
import os
import tempfile

folder_path = "/kaggle/working"
zip_name = os.path.join(folder_path, "Guftagu_1_to_28")

with tempfile.TemporaryDirectory() as temp_dir:
    
    # Copy only required files
    for i in range(1, 29):
        file_name = f"Guftagu_{i}.txt"
        src = os.path.join(folder_path, file_name)

        if os.path.exists(src):
            shutil.copy(src, temp_dir)

    # Make zip
    shutil.make_archive(zip_name, "zip", temp_dir)

print("ZIP created:", zip_name + ".zip")

ZIP created: /kaggle/working/Guftagu_1_to_28.zip
